# Week 1 — Data Acquisition, Cleaning, and Exploratory Data Analysis

**Project:** Iris Dataset Analysis  
**Author:** Vikas Yadav

This notebook demonstrates a complete data preparation workflow: acquiring a public dataset, inspecting data quality, simulating common data-quality issues for practice, cleaning and preprocessing the data, performing exploratory data analysis, creating visualizations, and documenting insights.

## Objectives
- Acquire a publicly available dataset.
- Inspect structure, data types, missing values, and duplicates.
- Demonstrate a reproducible cleaning workflow.
- Perform exploratory analysis with summary statistics.
- Create visualizations for distributions, relationships, and correlations.
- Summarize findings and identify possible next steps for machine learning.


## 1. Dataset Source

The Iris dataset is from the **UCI Machine Learning Repository**. It contains 150 observations, four numerical measurements, and three iris species.

**Official source:** https://archive.ics.uci.edu/dataset/53/iris

> For this Week 1 exercise, controlled missing values and duplicate rows are introduced into a working copy so that the notebook demonstrates realistic cleaning operations. The original UCI dataset itself is a clean benchmark dataset.


In [ ]:
# Install dependencies if needed:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")


## 2. Acquire the Dataset


In [ ]:
iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=[
        "sepal_length_cm",
        "sepal_width_cm",
        "petal_length_cm",
        "petal_width_cm",
    ],
)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Initial Data Inspection


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nSummary statistics:")
display(df.describe(include="all"))

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 4. Simulate Data-Quality Problems

Five numerical values are intentionally replaced with missing values and three additional duplicate rows are appended. This creates a realistic working dataset while keeping the original source data unchanged.


In [ ]:
work_df = df.copy()

missing_cells = [
    (10, "sepal_length_cm"),
    (25, "sepal_width_cm"),
    (60, "petal_length_cm"),
    (90, "petal_width_cm"),
    (120, "sepal_length_cm"),
]

for row_idx, column in missing_cells:
    work_df.loc[row_idx, column] = np.nan

work_df = pd.concat([work_df, work_df.iloc[[0, 50, 100]]], ignore_index=True)

print("Working dataset shape:", work_df.shape)
print("Missing cells:", int(work_df.isna().sum().sum()))
print("Duplicate rows detected:", int(work_df.duplicated().sum()))
display(work_df.head())


### Note on duplicate detection

The duplicate count may be greater than the three rows explicitly added because the Iris data representation can already contain a repeated complete record. `duplicated()` reports all rows that are duplicates of an earlier row.


## 5. Missing-Value Analysis Before Cleaning


In [ ]:
missing_before = work_df.isna().sum()
display(missing_before[missing_before > 0].to_frame("missing_count"))

plt.figure(figsize=(9, 5))
missing_before.plot(kind="bar")
plt.title("Missing Values Before Cleaning")
plt.xlabel("Column")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 6. Cleaning and Preprocessing

### Cleaning decisions
1. Numerical columns are converted to numeric values.
2. Missing numerical values are imputed with the median.
3. Exact duplicate rows are removed.
4. Species is stored as a categorical variable.


In [ ]:
clean_df = work_df.copy()

numeric_cols = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]

clean_df[numeric_cols] = clean_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

for column in numeric_cols:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

clean_df = clean_df.drop_duplicates().reset_index(drop=True)
clean_df["species"] = clean_df["species"].astype("category")

print("Cleaned shape:", clean_df.shape)
print("Remaining missing cells:", int(clean_df.isna().sum().sum()))
print("Remaining duplicate rows:", int(clean_df.duplicated().sum()))
display(clean_df.head())


## 7. Missing Values After Cleaning


In [ ]:
missing_after = clean_df.isna().sum()
comparison = pd.DataFrame({"Before": missing_before, "After": missing_after})
display(comparison)

ax = comparison.plot(kind="bar", figsize=(9, 5))
ax.set_title("Missing Values: Before vs After Cleaning")
ax.set_xlabel("Column")
ax.set_ylabel("Number of Missing Values")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Summary Statistics


In [ ]:
summary_stats = clean_df[numeric_cols].describe().T
display(summary_stats)

print("Class counts:")
display(clean_df["species"].value_counts().sort_index().to_frame("count"))


## 9. Visualization — Species/Class Distribution


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=clean_df, x="species")
plt.title("Iris Species Distribution")
plt.xlabel("Species")
plt.ylabel("Number of Observations")
plt.tight_layout()
plt.show()


### Insight
The classes are approximately balanced, so no species dominates the dataset.


## 10. Visualization — Petal Length vs Petal Width


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=clean_df,
    x="petal_length_cm",
    y="petal_width_cm",
    hue="species",
    s=70,
)
plt.title("Petal Length vs Petal Width by Species")
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.legend(title="Species")
plt.tight_layout()
plt.show()


### Insight
Petal length and petal width show a strong relationship. Setosa forms a clearly separated cluster, while Versicolor and Virginica have some overlap.


## 11. Visualization — Correlation Heatmap


In [ ]:
corr = clean_df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()


### Insight
Petal length and petal width are strongly positively correlated. Correlated predictors should be considered when selecting features for later models.


## 12. Optional Pairwise Exploration


In [ ]:
sns.pairplot(clean_df, hue="species", corner=True)
plt.suptitle("Pairwise Relationships Across Iris Measurements", y=1.02)
plt.show()


## 13. Export the Cleaned Dataset


In [ ]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "iris_cleaned.csv")
clean_df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


## 14. Key Findings

- The dataset contains four numerical measurements and three species classes.
- The simulated missing values were identified and median-imputed.
- Duplicate records were identified and removed.
- The cleaned dataset contains no missing values or duplicate rows.
- The species classes are balanced.
- Petal length and petal width show a strong positive relationship.
- Setosa is visually distinct in several feature combinations, while Versicolor and Virginica overlap more.

## 15. Future Analysis

Possible next steps:
- Standardize features for distance-based algorithms.
- Train Logistic Regression, KNN, Decision Tree, and SVM classifiers.
- Use cross-validation for model evaluation.
- Compare accuracy, precision, recall, and F1-score.
- Apply PCA for dimensionality reduction.
- Investigate feature importance and model interpretability.

## Conclusion

This notebook demonstrates a reproducible Week 1 data-science workflow from acquisition and data-quality inspection through cleaning, exploratory analysis, visualization, and preparation for future machine-learning tasks.
